# **IMPORT LIBRARY**

In [68]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
import re
import string

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# **DATA LOADING**

In [69]:
print("DATA GAMES")
df_games = pd.read_csv("../data/raw/games.csv")
display(df_games.head())
print("\nInfo Data Games: ")
df_games.info()

DATA GAMES


,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
0,13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,0.0,True
1,22364,BRINK: Agents of Change,2011-08-03,True,False,False,Positive,85,21,2.99,2.99,0.0,True
2,113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,0.0,True
3,226560,Escape Dead Island,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,0.0,True
4,249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,0.0,True



Info Data Games: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50872 entries, 0 to 50871
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   app_id          50872 non-null  int64  
 1   title           50872 non-null  object 
 2   date_release    50872 non-null  object 
 3   win             50872 non-null  bool   
 4   mac             50872 non-null  bool   
 5   linux           50872 non-null  bool   
 6   rating          50872 non-null  object 
 7   positive_ratio  50872 non-null  int64  
 8   user_reviews    50872 non-null  int64  
 9   price_final     50872 non-null  float64
 10  price_original  50872 non-null  float64
 11  discount        50872 non-null  float64
 12  steam_deck      50872 non-null  bool   
dtypes: bool(4), float64(3), int64(3), object(3)
memory usage: 3.7+ MB


In [70]:
print("DATA REKOMENDASI")
df_recom = pd.read_csv("../data/raw/recommendations.csv")
display(df_recom.head())
print("\nInfo Data Rekomendasi: ")
df_recom.info()

DATA REKOMENDASI


,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,975370,0,0,2022-12-12,True,36.3,51580,0
1,304390,4,0,2017-02-17,False,11.5,2586,1
2,1085660,2,0,2019-11-17,True,336.5,253880,2
3,703080,0,0,2022-09-23,True,27.4,259432,3
4,526870,0,0,2021-01-10,True,7.9,23869,4



Info Data Rekomendasi: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41154794 entries, 0 to 41154793
Data columns (total 8 columns):
 #   Column          Dtype  
---  ------          -----  
 0   app_id          int64  
 1   helpful         int64  
 2   funny           int64  
 3   date            object 
 4   is_recommended  bool   
 5   hours           float64
 6   user_id         int64  
 7   review_id       int64  
dtypes: bool(1), float64(1), int64(5), object(1)
memory usage: 2.2+ GB


In [71]:
print("DATA METADATA")
df_meta = pd.read_json('../data/raw/games_metadata.json', lines=True)

display(df_meta.head())
print("\nInfo Data Metadata:")
df_meta.info()

DATA METADATA


,app_id,description,tags
0,13500,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
1,22364,,[Action]
2,113020,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
3,226560,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
4,249050,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."



Info Data Metadata:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50872 entries, 0 to 50871
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   app_id       50872 non-null  int64 
 1   description  50872 non-null  object
 2   tags         50872 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


# **DATA PREPROCESSING**

In [72]:
def cleaningText(text):
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = re.sub(r'#\w+', '', text)  # Remove hashtags
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)  # Remove non-alphanumeric characters

    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = text.replace('\n', ' ')  # Remove newlines
    text = text.strip(' ') # Remove leading and trailing whitespace
    return text

def caseFolding(text):
    text = text.lower()
    return text

def tokenizingText(text):
    text = word_tokenize(text)
    return text

def filteringText(text):
    listStopwords = set(stopwords.words('english'))

    filtered_text = []
    for txt in text:
        if txt not in listStopwords:
            filtered_text.append(txt)

    text = filtered_text
    return text

def stemmingText(tokens):
    stemmer = PorterStemmer()
    return [stemmer.stem(token) for token in tokens]

def lemmatizingText(tokens):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in tokens]
def toSentence(list_words):
    sentence = ' '.join(word for word in list_words)
    return sentence

### **Preprocessing Game Description**

In [73]:
df_meta = df_meta.applymap(lambda x: tuple(x) if isinstance(x, list) else x)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9680\2900696381.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_meta = df_meta.applymap(lambda x: tuple(x) if isinstance(x, list) else x)


In [74]:
clean_df = df_meta.dropna()

In [75]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50872 entries, 0 to 50871
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   app_id       50872 non-null  int64 
 1   description  50872 non-null  object
 2   tags         50872 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


In [76]:
clean_df = clean_df.drop_duplicates()

In [77]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50872 entries, 0 to 50871
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   app_id       50872 non-null  int64 
 1   description  50872 non-null  object
 2   tags         50872 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


In [78]:
# Preprocessing Metadata Description Text
clean_df['desc_clean'] = clean_df['description'].apply(cleaningText)
clean_df['desc_caseFolding'] = clean_df['desc_clean'].apply(caseFolding)
clean_df['desc_tokenizing'] = clean_df['desc_caseFolding'].apply(tokenizingText)
clean_df['desc_stopword'] = clean_df['desc_tokenizing'].apply(filteringText)
clean_df['desc_stemming'] = clean_df['desc_stopword'].apply(stemmingText)
clean_df['desc_lemmatizing'] = clean_df['desc_stopword'].apply(lemmatizingText)
clean_df['desc_sentence'] = clean_df['desc_lemmatizing'].apply(toSentence)

In [79]:
clean_df 

,app_id,description,tags,desc_clean,desc_caseFolding,desc_tokenizing,desc_stopword,desc_stemming,desc_lemmatizing,desc_sentence
0,13500,Enter the dark underworld of Prince of Persia ...,"(Action, Adventure, Parkour, Third Person, Gre...",Enter the dark underworld of Prince of Persia ...,enter the dark underworld of prince of persia ...,"[enter, the, dark, underworld, of, prince, of,...","[enter, dark, underworld, prince, persia, warr...","[enter, dark, underworld, princ, persia, warri...","[enter, dark, underworld, prince, persia, warr...",enter dark underworld prince persia warrior wi...
1,22364,,"(Action,)",,,[],[],[],[],
2,113020,Monaco: What's Yours Is Mine is a single playe...,"(Co-op, Stealth, Indie, Heist, Local Co-Op, St...",Monaco Whats Yours Is Mine is a single player ...,monaco whats yours is mine is a single player ...,"[monaco, whats, yours, is, mine, is, a, single...","[monaco, whats, mine, single, player, coop, he...","[monaco, what, mine, singl, player, coop, heis...","[monaco, whats, mine, single, player, coop, he...",monaco whats mine single player coop heist gam...
3,226560,Escape Dead Island is a Survival-Mystery adven...,"(Zombies, Adventure, Survival, Action, Third P...",Escape Dead Island is a SurvivalMystery advent...,escape dead island is a survivalmystery advent...,"[escape, dead, island, is, a, survivalmystery,...","[escape, dead, island, survivalmystery, advent...","[escap, dead, island, survivalmysteri, adventu...","[escape, dead, island, survivalmystery, advent...",escape dead island survivalmystery adventure l...
4,249050,Dungeon of the Endless is a Rogue-Like Dungeon...,"(Roguelike, Strategy, Tower Defense, Pixel Gra...",Dungeon of the Endless is a RogueLike DungeonD...,dungeon of the endless is a roguelike dungeond...,"[dungeon, of, the, endless, is, a, roguelike, ...","[dungeon, endless, roguelike, dungeondefense, ...","[dungeon, endless, roguelik, dungeondefens, ga...","[dungeon, endless, roguelike, dungeondefense, ...",dungeon endless roguelike dungeondefense game ...
...,...,...,...,...,...,...,...,...,...,...
50867,2296380,,(),,,[],[],[],[],
50868,1272080,,(),,,[],[],[],[],
50869,1402110,,(),,,[],[],[],[],
50870,2272250,Embark on a journey into the darkest nightmare...,"(Early Access, FPS, Action, Retro, First-Perso...",Embark on a journey into the darkest nightmare...,embark on a journey into the darkest nightmare...,"[embark, on, a, journey, into, the, darkest, n...","[embark, journey, darkest, nightmares, restore...","[embark, journey, darkest, nightmar, restor, s...","[embark, journey, darkest, nightmare, restore,...",embark journey darkest nightmare restore sanit...


In [81]:
clean_df.to_csv('C:\\Users\\LENOVO\\Desktop\\kulyeahhhhhh\\Pijak\\dataPreprocessing\\machine-learning\\notebooks\\cleaned_games_metadata.csv', index=False)